# Item 4: Cold-Start SFT on Tesla T4 Silicon

Train `Qwen2.5-Math-1.5B` base on the certified 605-seed bootstrap dataset (`data/chalk_seeds_500.jsonl`)
with LoRA (all 7 linear projections) and strict prompt loss masking.

**Hardware**: Free Tesla T4 GPU in Google Colab (16 GB GDDR6).
**Runtime**: Change runtime type -> T4 GPU -> Run all.

In [ ]:
# 1. Verify Tesla T4 GPU
!nvidia-smi
import torch
assert torch.cuda.is_available(), "CUDA GPU not detected! Change runtime type to T4 GPU."
print("Device:", torch.cuda.get_device_name(0))
print("VRAM:", f"{torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

In [ ]:
# 2. Clone repository or checkout branch
import os
if not os.path.exists("t4-cuda"):
    !git clone -b feat-cold-start-sft-math-1.5b https://github.com/Epoch-AI-Lab/t4-cuda.git
    %cd t4-cuda
else:
    %cd t4-cuda
    !git fetch origin
    !git checkout feat-cold-start-sft-math-1.5b
    !git pull origin feat-cold-start-sft-math-1.5b

In [ ]:
# 3. Install dependencies & custom T4 CUDA kernels
!pip install -q transformers peft sympy datasets accelerate pytest
!cd src && pip install -e . --quiet

In [ ]:
# 4. Run verification unit tests
!python3 -m pytest tests/test_math_sft_pipeline.py -v

In [ ]:
# 5. Execute SFT on Tesla T4 silicon
!python3 benchmarks/train_math_sft.py \
    --model_name_or_path Qwen/Qwen2.5-Math-1.5B \
    --data_path data/chalk_seeds_500.jsonl \
    --output_dir results/chalk_math_1.5b_sft \
    --epochs 3 \
    --batch_size 2 \
    --grad_accum 8 \
    --lr 2e-4 \
    --max_length 2048 \
    --lora_r 32 \
    --lora_alpha 64 \
    --logging_steps 5

In [ ]:
# 6. Evaluate on Held-Out Contest Math Benchmark (AMC 12 & AIME)
!python3 benchmarks/eval_math_benchmark.py \
    --model_name_or_path Qwen/Qwen2.5-Math-1.5B \
    --adapter_path results/chalk_math_1.5b_sft/lora_adapter \
    --benchmark_path data/external_math_eval.json \
    --output_path results/external_eval_results.json \
    --use_kernels \
    --device cuda